# Jetbot 初始化 + 系统自检

这个 Notebook 会完成：

1. 使用默认公开 sudo 密码。
2. 临时停止官方服务 `jetbot_start`。
3. 重启相机服务 `nvargus-daemon`。
4. 检查 Python、服务状态、网络、关键文件、Robot API、Camera API。
5. 保存 JSON/TXT 报告和一张相机测试照片。

说明：停止 `jetbot_start` 只是本次运行的临时操作，不会永久禁用官方 APP 服务。

In [ ]:
# 1. 配置项
RUN_INITIALIZATION = True
MOTION_TEST = False

CAMERA_WIDTH = 224
CAMERA_HEIGHT = 224
IMAGE_PATH = "init_selfcheck_frame.jpg"

INTERNET_URL = "https://www.baidu.com"
JSON_REPORT = "jetbot_init_selfcheck_report.json"
TEXT_REPORT = "jetbot_init_selfcheck_report.txt"

# 如果 MOTION_TEST=True，小车会短距离前进。默认关闭。
MOTION_SPEED = 0.35
MOTION_DURATION = 0.35


In [ ]:
# 2. sudo 密码
# Yahboom Jetbot 默认公开密码。密码不会写入报告文件。

SUDO_PASSWORD = "yahboom"

if not RUN_INITIALIZATION:
    print("跳过初始化，只运行自检。")
else:
    print("将使用默认公开 sudo 密码执行初始化。")


In [ ]:
# 3. 执行初始化 + 系统自检
import json
from jetbot_init_and_selfcheck import build_report, write_text_report

class Args(object):
    pass

args = Args()
args.skip_init = not RUN_INITIALIZATION
args.wait_seconds = 5
args.camera_width = CAMERA_WIDTH
args.camera_height = CAMERA_HEIGHT
args.image_path = IMAGE_PATH
args.internet_url = INTERNET_URL
args.internet_timeout = 8
args.motion_test = MOTION_TEST
args.motion_speed = MOTION_SPEED
args.motion_duration = MOTION_DURATION

report = build_report(args, SUDO_PASSWORD)

with open(JSON_REPORT, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
write_text_report(report, TEXT_REPORT)

print(report["summary"]["message"])
print("JSON report:", JSON_REPORT)
print("Text report:", TEXT_REPORT)
if report["checks"].get("camera", {}).get("image_saved"):
    print("Camera frame:", report["checks"]["camera"].get("image_path"))

report


In [ ]:
# 4. 显示相机测试照片
from IPython.display import Image, display

image_path = report.get("checks", {}).get("camera", {}).get("image_path")
print("image_path =", image_path)
if image_path:
    display(Image(filename=image_path))


## 可选运动测试

如需测试运动，把第一段配置中的 `MOTION_TEST = True`，并确保小车前方安全或履带架空。默认不会运动。